# Библиотеки

In [ ]:
!pip install datasets

In [ ]:
! pip install seqeval

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_from_disk
from transformers import AutoTokenizer
from seqeval.metrics import f1_score
from transformers import AutoModelForTokenClassification, AutoConfig
from transformers import Trainer, TrainingArguments
from transformers import BertConfig, BertModel
from transformers.modeling_outputs import TokenClassifierOutput
from transformers import get_scheduler
from tqdm import tqdm
from seqeval.metrics import classification_report, f1_score

# Данные

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset = load_from_disk("/content/drive/My Drive/Colab Notebooks/DL/conll2003")

dataset

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 14041
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 3453
    })
})

In [ ]:
dataset['train'][0]

{'words': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [ ]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [ ]:
words = dataset["train"][0]["words"]
labels = dataset["train"][0]["tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


# Предобработка

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
words = dataset["train"][0]["words"]
inputs = tokenizer(words, is_split_into_words=True)

print('Слова: ', words)
print('Токены:', inputs.tokens())

Слова:  ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Токены: ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']


In [ ]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = np.full(len(word_ids), -100)  # -100 – специальное значение
    current_word = None
    for i, word_id in enumerate(word_ids[1:-1]):
        # следующее слово
        if word_id != current_word:
            current_word = word_id
            new_labels[i+1] = labels[word_id]
        # то же слово
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1  # меняем B- на I-
            new_labels[i+1] = label

    return new_labels

In [ ]:
def tokenize_and_align(batch):
    # Токенизация и выравнивание батча текстов
    tokenized = tokenizer(batch["words"], is_split_into_words=True)
    all_labels = batch["tags"]
    aligned_labels = []
    for i, labels in enumerate(all_labels):
        aligned = align_labels_with_tokens(labels, tokenized.word_ids(i))
        aligned_labels.append(aligned)

    return {
        'input_ids': tokenized['input_ids'],
        'labels': aligned_labels
    }

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset['train'].column_names,
)

# Метрика

In [ ]:
def compute_f1(data):
    logits, labels = data
    predictions = np.argmax(logits, axis=-1)

    # Удаляем специальные токены и преобразуем в текстовые метки
    text_labels = []
    text_predictions = []
    for i in range(len(labels)):
        named_labels = []
        named_preds = []
        for j in range(labels.shape[1]):
            if labels[i, j] != -100:
                named_labels.append(label_names[labels[i, j]])
                named_preds.append(label_names[predictions[i, j]])

        text_labels.append(named_labels)
        text_predictions.append(named_preds)

    return {'f1': f1_score(text_labels, text_predictions)}

# Модель

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
config = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(label_names))
model = AutoModelForTokenClassification.from_config(config).to(device)

# Сначала загружаем state_dict с указанием map_location
state_dict = torch.load('/content/drive/My Drive/Colab Notebooks/DL/bert-base-cased-ner.pt', map_location=torch.device('cpu'))

# Загружаем параметры в модель
model.load_state_dict(state_dict)

print('Число параметров:', sum(p.numel() for p in model.parameters()))


Число параметров: 107726601


# Факторизация матрицы эмбеддингов

In [ ]:
class FactorizedEmbedding(nn.Module):
    def __init__(self, embedding: nn.Embedding, rank: int = 64):
        super().__init__()

        weights = embedding.weight.data
        u, s, v = torch.pca_lowrank(weights, q=rank, center=False, niter=10)

        self.embedding = nn.Sequential(
            nn.Embedding(weights.shape[0], rank), nn.Linear(rank, weights.shape[1], bias=False)
        ).to(weights.device)

        self.embedding[0].weight.data = u @ torch.diag(s)
        self.embedding[1].weight.data = v

    def forward(self, input_ids):
        return self.embedding(input_ids)


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

In [ ]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
model.bert.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(28996, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [ ]:
# Заменяем word_embeddings на факторизованный
model.bert.embeddings.word_embeddings = FactorizedEmbedding(model.bert.embeddings.word_embeddings, rank=64)

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    num_train_epochs=20,
    output_dir='./ner-factorized',
    report_to='none',
    eval_strategy='epoch',
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_f1,
)

trainer.train()

# Проверка на тесте
trainer.evaluate(tokenized_datasets["test"])

/tmp/ipython-input-3783814230.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.020400,0.276863,0.870171
2,0.012600,0.268953,0.869209
3,0.008300,0.269518,0.875479
4,0.013000,0.268951,0.877162
5,0.008100,0.279530,0.869128
6,0.005100,0.285460,0.876536
7,0.004000,0.303299,0.884236
8,0.003600,0.331618,0.877921
9,0.002700,0.314656,0.876300
10,0.002400,0.329575,0.879637


{'eval_loss': 0.3552769422531128,
 'eval_f1': 0.8885967218862302,
 'eval_runtime': 3.3955,
 'eval_samples_per_second': 1016.931,
 'eval_steps_per_second': 63.613,
 'epoch': 20.0}

In [ ]:
grader_dataset = load_from_disk("/content/drive/My Drive/Colab Notebooks/DL/grader_conll2003")

tokenized_grader_dataset = grader_dataset.map(
    lambda sample: tokenizer(sample["words"], is_split_into_words=True),
    remove_columns=grader_dataset.column_names,
)

In [ ]:
@torch.inference_mode()
def predict_and_save(model, dataset, file_path='/content/drive/My Drive/Colab Notebooks/DL/predictions.txt'):
    model.eval()
    with open(file_path, 'w') as f:
        for sample in dataset:
            logits = model(
                torch.tensor([sample['input_ids']], device=device)
            ).logits.cpu().squeeze(0)

            tags = logits.argmax(axis=-1)[1:-1]
            tag_names = [label_names[tag] for tag in tags]

            f.write(' '.join(tag_names))
            f.write('\n')

In [ ]:
predict_and_save(model, tokenized_grader_dataset)

# Дистилляция

In [ ]:
config = AutoConfig.from_pretrained('bert-base-cased', num_labels=9)
config

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.56.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        config = AutoConfig.from_pretrained('bert-base-cased', num_labels=9)

        config.intermediate_size = 948
        config.num_hidden_layers = 6
        config.hidden_size = 512
        config.num_attention_heads = 8

        self.model = AutoModelForTokenClassification.from_config(config)
        self.model.bert.embeddings.word_embeddings = FactorizedEmbedding(
            self.model.bert.embeddings.word_embeddings, rank=256
        )

    def forward(self, input_ids, attention_mask=None):
        return self.model(input_ids, attention_mask)


student_model = Model()
student_model.to(device)
teacher_model = model
teacher_model.to(device)
teacher_model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): FactorizedEmbedding(
        (embedding): Sequential(
          (0): Embedding(28996, 64)
          (1): Linear(in_features=64, out_features=768, bias=False)
        )
      )
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSel

In [ ]:
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)

In [ ]:
total_trainable_params = sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_trainable_params}")


Total trainable parameters: 0


In [ ]:
sum(p.numel() for p in student_model.parameters())

19972161

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=2, ignore_index=-100):
    ce_loss_fct = nn.CrossEntropyLoss(ignore_index=ignore_index)

    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=-1),
        F.softmax(teacher_logits / temperature, dim=-1),
        reduction='batchmean'
    ) * (temperature ** 2)

    hard_loss = ce_loss_fct(student_logits.view(-1, student_logits.size(-1)), labels.view(-1))

    loss = soft_loss + hard_loss
    return loss


In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    tokenized_datasets["train"],
    batch_size=16,
    collate_fn=data_collator,
)
test_dataloader = torch.utils.data.DataLoader(
    tokenized_datasets["test"],
    batch_size=16,
    collate_fn=data_collator,
)


In [ ]:
teacher_num_labels = None
student_num_labels = None

for name, module in teacher_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        teacher_num_labels = module.out_features

for name, module in student_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        student_num_labels = module.out_features

assert teacher_num_labels == student_num_labels, f"Mismatch num_labels: teacher={teacher_num_labels}, student={student_num_labels}"
print(f"Both have num_labels={teacher_num_labels}")


Both have num_labels=9


In [ ]:
optimizer = torch.optim.AdamW(student_model.parameters(), lr=3e-5)
num_training_steps = len(train_dataloader) * 3
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.05 * num_training_steps),
    num_training_steps=num_training_steps,
)

for epoch in range(10):
    student_model.train()
    total_loss = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Учитель: логиты без градиентов
        with torch.no_grad():
            teacher_logits = teacher_model(input_ids=input_ids, attention_mask=attention_mask).logits

        # Ученик
        student_logits = student_model(input_ids=input_ids, attention_mask=attention_mask)

        # Потери
        loss = distillation_loss(student_logits.logits, teacher_logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {total_loss / len(train_dataloader):.4f}")

Epoch 1: 100%|██████████| 878/878 [00:41<00:00, 21.16it/s]


Epoch 1 - Loss: 88.7596


Epoch 2: 100%|██████████| 878/878 [00:40<00:00, 21.63it/s]


Epoch 2 - Loss: 41.6295


Epoch 3: 100%|██████████| 878/878 [00:41<00:00, 21.17it/s]


...

Epoch 8 - Loss: 29.6831


Epoch 9: 100%|██████████| 878/878 [00:40<00:00, 21.68it/s]


Epoch 9 - Loss: 29.4711


Epoch 10: 100%|██████████| 878/878 [00:40<00:00, 21.63it/s]

Epoch 10 - Loss: 29.5844


In [ ]:
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

def get_predictions(model, dataloader, device):
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        labels = labels.numpy()

        for pred, label in zip(predictions, labels):
            true_labels = []
            pred_labels = []

            for p, l in zip(pred, label):
                if l != -100:
                    true_labels.append(label_list[l])
                    pred_labels.append(label_list[p])

            all_labels.append(true_labels)
            all_preds.append(pred_labels)

    return all_preds, all_labels


# Получаем предсказания
preds, refs = get_predictions(student_model, test_dataloader, device)

# Печатаем метрики
print(classification_report(refs, preds))
print(f"F1: {f1_score(refs, preds):.4f}")

Evaluating: 100%|██████████| 216/216 [00:02<00:00, 85.55it/s]


              precision    recall  f1-score   support

         LOC       0.78      0.81      0.80      1668
        MISC       0.61      0.65      0.63       702
         ORG       0.63      0.69      0.66      1661
         PER       0.64      0.70      0.67      1617

   micro avg       0.67      0.72      0.70      5648
   macro avg       0.66      0.71      0.69      5648
weighted avg       0.67      0.72      0.70      5648

F1: 0.6964
